In [ ]:
import zipfile
import os
import requests

# Caminho do arquivo ZIP
zip_url = "https://caelum-online-public.s3.amazonaws.com/challenge-spark/semana-1.zip"
zip_path = "/content/semana-1.zip"
extract_path = "/content/semana-1/"

# Baixar o arquivo ZIP
response = requests.get(zip_url)
with open(zip_path, "wb") as f:
    f.write(response.content)

# Extrair o conteúdo
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

# Listar os arquivos extraídos
os.listdir(extract_path)

['dataset_bruto.json']

In [ ]:
from pyspark.sql import SparkSession

# Inicializar o Spark
spark = SparkSession.builder.appName("ProcessamentoJSON").getOrCreate()

# Caminho do arquivo JSON
json_path = "/content/semana-1/dataset_bruto.json"  # Substitua pelo nome real do arquivo

# Carregar o arquivo JSON em um DataFrame
df = spark.read.json(json_path)

# Exibir as primeiras linhas
df.show(10)

+--------------------+--------------------+--------------------+
|             anuncio|             imagens|             usuario|
+--------------------+--------------------+--------------------+
|{0, [], [16], [0]...|[{39d6282a-71f3-4...|{9d44563d-3405-4e...|
|{0, [], [14], [0]...|[{23d2b3ab-45b0-4...|{36245be7-70fe-40...|
|{0, [1026], [1026...|[{1da65baa-368b-4...|{9dc415d8-1397-4d...|
|{0, [120], [120],...|[{79b542c6-49b4-4...|{9911a2df-f299-4a...|
|{0, [3], [3], [0]...|[{e2bc497b-6510-4...|{240a7aab-12e5-40...|
|{0, [20], [15], [...|[{2de09d46-dc0d-4...|{3c7057f5-0923-42...|
|{3, [43], [43], [...|[{147a80d9-cd40-4...|{5a9736b5-aaa0-4a...|
|{2, [42], [42], [...|[{35740004-063d-4...|{ec48d96a-137c-49...|
|{0, [], [12], [0]...|[{6d3d2aec-c96f-4...|{dad7db63-e19c-44...|
|{1, [41], [41], [...|[{3d404069-418e-4...|{a845f35f-3ab3-46...|
+--------------------+--------------------+--------------------+
only showing top 10 rows



# Para nossa análise e tratamentos dos dados, a equipe solicitou que apenas a coluna "anuncio" será utilizado. Logo faremos a extração dela e transformaremos em um novo DataFrame

In [ ]:
tabela = df.select('anuncio')
arrays_df = tabela.collect()
arrays_df[0]

Row(anuncio=Row(andar=0, area_total=[], area_util=['16'], banheiros=[0], caracteristicas=[], endereco=Row(bairro='Centro', cep='20061003', cidade='Rio de Janeiro', estado='Rio de Janeiro', latitude=-22.906082, longitude=-43.18671, pais='BR', rua='Rua Buenos Aires', zona='Zona Central'), id='47d553e0-79f2-4a46-9390-5a3c962740c2', quartos=[0], suites=[0], tipo_anuncio='Usado', tipo_unidade='Outros', tipo_uso='Comercial', vaga=[1], valores=[Row(condominio='260', iptu='107', tipo='Venda', valor='10000')]))

In [ ]:
elementos = [array['anuncio'] for array in arrays_df]

df_final = spark.createDataFrame(elementos)
df_final.show(5)

+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|andar|area_total|area_util|banheiros|     caracteristicas|            endereco|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|             valores|
+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|    0|        []|     [16]|      [0]|                  []|{Centro, 20061003...|47d553e0-79f2-4a4...|    [0]|   [0]|       Usado|      Outros|  Comercial| [1]|[{260, 107, Venda...|
|    0|        []|     [14]|      [0]|                  []|{Centro, 20051040...|b6ffbae1-17f6-487...|    [0]|    []|       Usado|      Outros|  Comercial| [0]|[{260, 107, Venda...|
|    0|    [1026]|   [1026]|      [0]|                  []|{Maria da Graça, ...|1fb030a5-9e3e-4

# O time de Data Science solicitou que fizéssemos alguns filtros nas colunas `tipo_uso`, `tipo_unidade` e `tipo_anuncio` da nossa base de dados:

- tipo_uso: **Residencial**;

- tipo_unidade: **Apartamento**;

- tipo_anuncio: **Usado**.

In [ ]:
df_final = df_final.select('*') \
                   .where((df_final['tipo_uso'] == 'Residencial') \
                           & (df_final['tipo_unidade'] == 'Apartamento') \
                           & (df_final['tipo_anuncio'] == 'Usado'))
df_final.show(5)

+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|andar|area_total|area_util|banheiros|     caracteristicas|            endereco|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|             valores|
+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|    3|      [43]|     [43]|      [1]|[Academia, Churra...|{Paciência, 23585...|d2e3a3aa-09b5-45a...|    [2]|    []|       Usado| Apartamento|Residencial| [1]|[{245, NULL, Vend...|
|    2|      [42]|     [42]|      [1]|[Churrasqueira, P...|{Paciência, 23585...|085bab2c-87ad-452...|    [2]|    []|       Usado| Apartamento|Residencial| [1]|[{0, 0, Venda, 15...|
|    1|      [41]|     [41]|      [1]|[Portaria 24h, Co...|{Guaratiba, 23036...|18d22cbe-1b86-4

In [ ]:
df_final.printSchema()

root
 |-- andar: long (nullable = true)
 |-- area_total: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- area_util: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- banheiros: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- caracteristicas: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- endereco: struct (nullable = true)
 |    |-- bairro: string (nullable = true)
 |    |-- cep: string (nullable = true)
 |    |-- cidade: string (nullable = true)
 |    |-- estado: string (nullable = true)
 |    |-- latitude: double (nullable = true)
 |    |-- longitude: double (nullable = true)
 |    |-- pais: string (nullable = true)
 |    |-- rua: string (nullable = true)
 |    |-- zona: string (nullable = true)
 |-- id: string (nullable = true)
 |-- quartos: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- suites: array (nullable = true)
 |    |-- element: long (c

# Como é percepitível, algumas colunas estão com estrutura de array e isso atrapalha se quisermos fazer um modelo de machine learning posteriormente. Nesse contexto, transformaremos os dados das colunas "quartos", "suites", "banheiros", "vaga", "area_total" e "area_util" de listas para inteiros.

In [ ]:
from pyspark.sql.types import IntegerType, StringType

In [ ]:
df_final = df_final.withColumn("quartos",df_final.quartos[0].cast(IntegerType()))
df_final = df_final.withColumn("suites",df_final.suites[0].cast(IntegerType()))
df_final = df_final.withColumn("banheiros",df_final.banheiros[0].cast(IntegerType()))
df_final = df_final.withColumn("vaga",df_final.vaga[0].cast(IntegerType()))
df_final = df_final.withColumn("area_total",df_final.area_total[0].cast(IntegerType()))
df_final = df_final.withColumn("area_util",df_final.area_util[0].cast(IntegerType()))

In [ ]:
df_final.printSchema()

root
 |-- andar: long (nullable = true)
 |-- area_total: integer (nullable = true)
 |-- area_util: integer (nullable = true)
 |-- banheiros: integer (nullable = true)
 |-- caracteristicas: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- endereco: struct (nullable = true)
 |    |-- bairro: string (nullable = true)
 |    |-- cep: string (nullable = true)
 |    |-- cidade: string (nullable = true)
 |    |-- estado: string (nullable = true)
 |    |-- latitude: double (nullable = true)
 |    |-- longitude: double (nullable = true)
 |    |-- pais: string (nullable = true)
 |    |-- rua: string (nullable = true)
 |    |-- zona: string (nullable = true)
 |-- id: string (nullable = true)
 |-- quartos: integer (nullable = true)
 |-- suites: integer (nullable = true)
 |-- tipo_anuncio: string (nullable = true)
 |-- tipo_unidade: string (nullable = true)
 |-- tipo_uso: string (nullable = true)
 |-- vaga: integer (nullable = true)
 |-- valores: array (nullable = true)
 

In [ ]:
df_final.show(5)

+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|andar|area_total|area_util|banheiros|     caracteristicas|            endereco|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|             valores|
+-----+----------+---------+---------+--------------------+--------------------+--------------------+-------+------+------------+------------+-----------+----+--------------------+
|    3|        43|       43|        1|[Academia, Churra...|{Paciência, 23585...|d2e3a3aa-09b5-45a...|      2|  NULL|       Usado| Apartamento|Residencial|   1|[{245, NULL, Vend...|
|    2|        42|       42|        1|[Churrasqueira, P...|{Paciência, 23585...|085bab2c-87ad-452...|      2|  NULL|       Usado| Apartamento|Residencial|   1|[{0, 0, Venda, 15...|
|    1|        41|       41|        1|[Portaria 24h, Co...|{Guaratiba, 23036...|18d22cbe-1b86-4

# Como as colunas "endereco" e "valores" estão compactadas, extrairemos em DataFrames diferentes para mais tarde tratamos delas

In [ ]:
import pyspark.sql.functions as f

In [ ]:
df_final = df_final.select('*', 'endereco.*').drop('endereco')

df_final = df_final.withColumn('valores', f.explode('valores'))
df_final = df_final.select('*', 'valores.*').drop('valores')

df_final.show(5)

+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+--------+--------------+--------------+----------+----------+----+--------------------+----------+----------+----+-----+-----+
|andar|area_total|area_util|banheiros|     caracteristicas|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|   bairro|     cep|        cidade|        estado|  latitude| longitude|pais|                 rua|      zona|condominio|iptu| tipo|valor|
+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+--------+--------------+--------------+----------+----------+----+--------------------+----------+----------+----+-----+-----+
|    3|        43|       43|        1|[Academia, Churra...|d2e3a3aa-09b5-45a...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|23585430|Rio de Janeiro|Rio

## A equipe de ciência de dados nos solicitou que apenas as informações sobre bairro e zona da cidade fossem extraídas. Logo retiraremos as colunas não solicitadas

In [ ]:
df_final = df_final.drop('cep')
df_final = df_final.drop('cidade')
df_final = df_final.drop('estado')
df_final = df_final.drop('latitude')
df_final = df_final.drop('longitude')
df_final = df_final.drop('pais')
df_final = df_final.drop('rua')

df_final.show(5)

+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|andar|area_total|area_util|banheiros|     caracteristicas|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|   bairro|      zona|condominio|iptu| tipo|valor|
+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|    3|        43|       43|        1|[Academia, Churra...|d2e3a3aa-09b5-45a...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|       245|NULL|Venda|15000|
|    2|        42|       42|        1|[Churrasqueira, P...|085bab2c-87ad-452...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|         0|   0|Venda|15000|
|    1|        41|       41|        1|[Portaria 24h, Co...|1

## A InsightPlaces permite que o(a) anunciante crie um anúncio com duas opções de valor. Assim, o(a) cliente pode criar um anúncio que mostre tanto o valor de venda do imóvel quanto o seu valor de locação, juntamente com os valores de taxa de condomínio (quando houver) e taxa de IPTU. Estes valores são diferenciados pelo campo tipo que pode assumir os valores Venda e Aluguel.

## Como se trata de um estudo sobre o preço de venda dos imóveis, o time de cientistas de dados solicitou apenas as informações do tipo VENDA. Logo faremos um filtro a partir desta coluna.

In [ ]:
df_final.select('tipo').distinct().show()

+-------+
|   tipo|
+-------+
|Aluguel|
|  Venda|
+-------+



In [ ]:
df_final = df_final.select('*').where(df_final['tipo'] == 'Venda')

df_final.select('tipo').distinct().show()

+-----+
| tipo|
+-----+
|Venda|
+-----+



In [ ]:
df_final.show(5)

+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|andar|area_total|area_util|banheiros|     caracteristicas|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|   bairro|      zona|condominio|iptu| tipo|valor|
+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|    3|        43|       43|        1|[Academia, Churra...|d2e3a3aa-09b5-45a...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|       245|NULL|Venda|15000|
|    2|        42|       42|        1|[Churrasqueira, P...|085bab2c-87ad-452...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|         0|   0|Venda|15000|
|    1|        41|       41|        1|[Portaria 24h, Co...|1

# Por fim salvaremos o arquivo em formato parquet e csv e compararemos os dois. Como csv não suporta o modelo de estrutura da coluna CARACTERISTICAS, para este caso transformaremos em uma STRING

In [ ]:
df_final.write.parquet('/content/df_imoveis.parquet')

In [ ]:
df_final = df_final.withColumn('caracteristicas', df_final['caracteristicas'].cast(StringType()))

df_final.write.csv('/content/df_imoveis.csv')

In [ ]:
%%time
df_parquet = spark.read.parquet('/content/df_imoveis.parquet')
df_parquet.show(5)

+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|andar|area_total|area_util|banheiros|     caracteristicas|                  id|quartos|suites|tipo_anuncio|tipo_unidade|   tipo_uso|vaga|   bairro|      zona|condominio|iptu| tipo|valor|
+-----+----------+---------+---------+--------------------+--------------------+-------+------+------------+------------+-----------+----+---------+----------+----------+----+-----+-----+
|    3|        43|       43|        1|[Academia, Churra...|d2e3a3aa-09b5-45a...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|       245|NULL|Venda|15000|
|    2|        42|       42|        1|[Churrasqueira, P...|085bab2c-87ad-452...|      2|  NULL|       Usado| Apartamento|Residencial|   1|Paciência|Zona Oeste|         0|   0|Venda|15000|
|    1|        41|       41|        1|[Portaria 24h, Co...|1

In [ ]:
%%time
df_csv = spark.read.csv('/content/df_imoveis.csv')
df_csv.show(5)

+---+---+---+---+--------------------+--------------------+---+----+-----+-----------+-----------+----+---------+----------+----+----+-----+-----+
|_c0|_c1|_c2|_c3|                 _c4|                 _c5|_c6| _c7|  _c8|        _c9|       _c10|_c11|     _c12|      _c13|_c14|_c15| _c16| _c17|
+---+---+---+---+--------------------+--------------------+---+----+-----+-----------+-----------+----+---------+----------+----+----+-----+-----+
|  3| 43| 43|  1|[Academia, Churra...|d2e3a3aa-09b5-45a...|  2|NULL|Usado|Apartamento|Residencial|   1|Paciência|Zona Oeste| 245|NULL|Venda|15000|
|  2| 42| 42|  1|[Churrasqueira, P...|085bab2c-87ad-452...|  2|NULL|Usado|Apartamento|Residencial|   1|Paciência|Zona Oeste|   0|   0|Venda|15000|
|  1| 41| 41|  1|[Portaria 24h, Co...|18d22cbe-1b86-476...|  2|NULL|Usado|Apartamento|Residencial|   1|Guaratiba|Zona Oeste|   0|   0|Venda|20000|
|  3| 43| 43|  1|[Churrasqueira, P...|bed8a354-9317-442...|  2|NULL|Usado|Apartamento|Residencial|   0|   Cosmos|Zona 

# Como é percepitível, além de o arquivo parquet carrega mais rápido, temos que no arquivo csv não é carregado os nomes das colunas, sendo dada a função para quem for mexer nos arquivos.